# ResNet-34 / CIFAR-100 --- the second dataset

Everything in the paper so far is CIFAR-10. This asks whether the effect
survives a 100-class task, on the one model with no published comparator row
(the original CAP work used ResNet-50/101), so there is no reproduction
pressure on these numbers --- they stand or fall on their own.

Full grid: 3 criteria x 4 sparsities x 3 seeds x 2 arms = 72 cells, plus two
dense baselines. Roughly 7 hours.

## The protocol was checked before this was launched

I.P.\ runs SGD 0.01 and BaCP 0.1, the CIFAR-10 split. That is an assumption on
a 100-class task --- a 100-way cross-entropy has a different loss scale than a
10-way one, which changes how the CE term competes with the three contrastive
terms --- and it is the assumption that failed on MobileNetV2, where an
inherited untuned rate turned a true +1.7 delta into a headline +70.3.

So it was measured first (`A0_cifar100_lr_probe`):

| | accuracy |
|---|---|
| dense | 72.67 |
| I.P.\ @ 0.95, **lr 0.01** | **70.27** |
| I.P.\ @ 0.95, lr 0.1 | 63.45 |

0.01 wins by 6.82 points. The split transfers, and the grid runs unmodified.

## What to expect, stated in advance

Pruning costs only **2.4 points** at 0.95 on this task (72.67 -> 70.27). That
is the VGG-11 signature: where gradual pruning does almost no damage, a
contrastive term added to an objective already at its ceiling has nothing to
repair, and the paper predicts approximately no gain. Expect small or absent
deltas at 0.95 and 0.97, with the informative cells at 0.99 and 0.999 --- on
CIFAR-10 this model gave +0.95 and +11.80 there.

A null at low sparsity is a result under that reading, not a failure, and it
is worth having measured rather than assumed.

## Ordering

Criterion-major with ascending sparsity: magnitude fully, then SNIP-it, then
WANDA. An interrupted run then leaves whole criteria at n=3 rather than three
fragments, and the criterion the paper leads with finishes first.

Safe to interrupt and re-run --- a recorded cell is skipped. The seed-1 dense
baseline from the probe is reused automatically.


In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Configuration and preflight

In [ ]:
MODEL, DATASET, NCLS = 'resnet34', 'cifar100', 100
SEEDS = (1, 2, 3)
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
PRUNERS = ('magnitude', 'snip', 'wanda')   # headline criterion first
GPU = 0

nb.fetch_imagenet_weights(MODEL)
nb.preflight(MODEL, num_classes=NCLS)

## Dense baselines

Every sparse cell resolves its checkpoint from the dense run of the SAME
seed. Seed 1 already exists from the learning-rate probe and skips.

In [ ]:
dense = [nb.make_cell(MODEL, 'dense', seed=s, dataset_name=DATASET, num_classes=NCLS)
         for s in SEEDS]
nb.run_group(dense, gpu=GPU)

## Plan

Built and sanity-checked before any training. The arms use the family
defaults --- I.P.\ 0.01, BaCP 0.1 --- which is what the probe validated, so
nothing is overridden here except the dataset.

In [ ]:
plan = []
for pruner in PRUNERS:
    for sp in SPARSITIES:
        for seed in SEEDS:
            plan.append(nb.make_cell(MODEL, 'prune', seed=seed, pruner=pruner,
                                     sparsity=sp, dataset_name=DATASET,
                                     num_classes=NCLS))
            plan.append(nb.make_cell(MODEL, 'bacp', seed=seed, pruner=pruner,
                                     sparsity=sp, dataset_name=DATASET,
                                     num_classes=NCLS))

n_ip = sum(1 for c in plan if c['rung'] == 'static-prune')
print(f'{len(plan)} cells = {n_ip} I.P. + {len(plan) - n_ip} BaCP')
print(f'est ~{(n_ip * 2.8 + (len(plan) - n_ip) * 8.0) / 60:.1f} h '
      f'(measured: resnet34 I.P. 2.8 min, BaCP 8.0 min)')

# the protocol the probe validated, asserted rather than trusted
for c in plan:
    cfg = c['config']
    assert cfg['dataset_name'] == DATASET, cfg['dataset_name']
    assert cfg['num_classes'] == NCLS, cfg['num_classes']
    want = 0.01 if c['rung'] == 'static-prune' else 0.1
    assert cfg['learning_rate'] == want, (c['key'], cfg['learning_rate'], want)

assert nb.sanity_check(plan), 'sanity check failed'

## Run

One row per epoch. `results.csv` is rewritten after every cell.

In [ ]:
nb.run_group(plan, gpu=GPU)

## Results --- CIFAR-100 beside CIFAR-10

In [ ]:
import json, glob, os, statistics as st
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k:
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

def cell(arm, pruner, sp, ds):
    xs = [acc.get(f'static.{arm}.{MODEL}.{ds}.s{sp}.{pruner}.seed{n}') for n in SEEDS]
    xs = [x for x in xs if x is not None]
    if not xs:
        return None
    return st.mean(xs), (st.stdev(xs) if len(xs) > 1 else 0.0), len(xs)

d100 = [acc.get(f'static.dense.{MODEL}.{DATASET}.dense.seed{n}') for n in SEEDS]
d100 = [x for x in d100 if x is not None]
if d100:
    sd = st.stdev(d100) if len(d100) > 1 else 0.0
    print(f'dense CIFAR-100  {st.mean(d100):.2f}+-{sd:.2f} (n={len(d100)})')

for pruner in PRUNERS:
    print()
    print(f'--- {pruner} ---')
    print(f'{"sp":>7} | {"C100 I.P.":>15} {"C100 BaCP":>15} {"d":>7} | {"C10 d":>7}')
    for sp in SPARSITIES:
        a = cell('prune', pruner, sp, DATASET)
        b = cell('bacp', pruner, sp, DATASET)
        x = cell('prune', pruner, sp, 'cifar10')
        y = cell('bacp', pruner, sp, 'cifar10')
        fa = f'{a[0]:.2f}+-{a[1]:.2f}' if a else '      --      '
        fb = f'{b[0]:.2f}+-{b[1]:.2f}' if b else '      --      '
        d = f'{b[0]-a[0]:+.2f}' if (a and b) else '   --'
        d10 = f'{y[0]-x[0]:+.2f}' if (x and y) else '   --'
        print(f'{sp:>7} | {fa:>15} {fb:>15} {d:>7} | {d10:>7}')

print()
print('C10 d is the same cell on CIFAR-10, for reference. Where CIFAR-100')
print('deltas are near zero at low sparsity, check the dense-to-I.P. drop:')
print('little damage means little for the objective to repair, which is the')
print('reading the paper already applies to VGG-11.')